In [4]:
# Cell 1: Imports and configuration
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import concurrent.futures
from tqdm.auto import tqdm
import re
import time
from datetime import datetime, timedelta
import logging

# Configuration
MODEL_NAME = "gemma3:12b"
# INPUT_CSV = "/data/gregIB/issuebench/2_final_dataset/Issues_Parties_combined.csv"
INPUT_CSV = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"
SAFE_MODEL_NAME = re.sub(r'[:/\\]', '-', MODEL_NAME)
OUTPUT_CSV = f"/data/gregIB/issuebench/3_experiments/2_inference/completions/020925_{SAFE_MODEL_NAME}_completions.csv"

# Model parameters
TEMPERATURE = 1
MAX_TOKENS = 1064
MAX_WORKERS = 8
MAX_RETRIES = 1
REQUEST_TIMEOUT = 300

# ETA configuration
ETA_UPDATE_FREQUENCY = 50

# Test mode
TEST_MODE = False
TEST_SAMPLE_SIZE = 50 if TEST_MODE else None

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [5]:
# Cell 2: OllamaProcessor class
class OllamaProcessor:
    """Optimized Ollama API processor with retry logic and error handling"""
    
    def __init__(self, model: str, max_workers: int = 4):
        self.model = model
        self.max_workers = max_workers
        self.session = self.create_session_with_retries()
        
    def create_session_with_retries(self):
        """Create a requests session with retry logic"""
        session = requests.Session()
        retry_strategy = Retry(
            total=MAX_RETRIES,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session
    
    def create_payload(self, prompt: str) -> dict:
        return {
            "model": self.model,
            "prompt": prompt.strip(),
            "think": True,
            "options": {
                "temperature": TEMPERATURE,
                # "num_predict": MAX_TOKENS
            },
            "stream": False
        }
    
    def process_single(self, index: int, prompt: str) -> tuple[int, str]:
        """Process a single prompt"""
        payload = self.create_payload(prompt)
        
        try:
            response = self.session.post(
                "http://localhost:11434/api/generate",
                json=payload,
                timeout=REQUEST_TIMEOUT
            )
            response.raise_for_status()
            return (index, response.json()['response'])
        except Exception as e:
            return (index, f"ERROR: {str(e)}")
    
    def process_batch(self, prompts: list[tuple[int, str]]) -> dict[int, str]:
        """Process a batch of prompts in parallel"""
        results = {}
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            # Submit all tasks
            future_to_index = {
                executor.submit(self.process_single, index, prompt): index
                for index, prompt in prompts
            }
            
            # Process completed tasks with progress bar
            completed_count = 0
            start_time = time.time()
            
            for future in tqdm(
                concurrent.futures.as_completed(future_to_index),
                total=len(prompts),
                desc="Processing prompts"
            ):
                index = future_to_index[future]
                try:
                    results[index] = future.result()[1]
                except Exception as e:
                    results[index] = f"ERROR: {str(e)}"
                
                # Update ETA periodically
                completed_count += 1
                if completed_count % ETA_UPDATE_FREQUENCY == 0 or completed_count == len(prompts):
                    elapsed_time = time.time() - start_time
                    if completed_count > 0:
                        avg_time_per_task = elapsed_time / completed_count
                        remaining_tasks = len(prompts) - completed_count
                        eta_seconds = avg_time_per_task * remaining_tasks
                        eta = str(timedelta(seconds=int(eta_seconds)))
                        logger.info(f"ETA: {eta} remaining")
        
        return results

In [ ]:
# Cell 3: Main execution
# Read and prepare data
logger.info("Reading input CSV...")
df = pd.read_csv(INPUT_CSV)

if TEST_MODE:
    df = df.head(TEST_SAMPLE_SIZE)
    logger.info(f"TEST MODE: Processing first {len(df)} rows")

# Initialize processor
processor = OllamaProcessor(model=MODEL_NAME, max_workers=MAX_WORKERS)

# Prepare prompts for processing
prompts = [(i, row['prompt_text']) for i, row in df.iterrows()]

# Process prompts
logger.info("Starting prompt processing...")
results = processor.process_batch(prompts)

# Update DataFrame with results
df['response_text'] = df.index.map(results)
df['model'] = MODEL_NAME

# Save results
logger.info(f"Saving results to {OUTPUT_CSV}")
df.to_csv(OUTPUT_CSV, index=False)

logger.info("Processing complete!")

2025-09-05 13:38:04,241 - INFO - Reading input CSV...
2025-09-05 13:38:06,670 - INFO - Starting prompt processing...
Processing prompts:   1%|▏         | 805/62178 [00:14<18:41, 54.73it/s]
